# Step 04. Hybrid A* 알고리즘

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

A\*는 격자 인덱스 (ix,iy)로 상태를 표현해 경로가 꺾입니다.  
Hybrid A\*는 상태를 연속 **(x, y, yaw)** 로 확장하고, 이웃 확장에 **Bicycle Model**을 써서 차량이 실제로 주행 가능한 부드러운 경로를 찾습니다.

> 작은 조각으로 나눠 이해하고, 마지막에 **격자 경로 vs Hybrid 경로**를 비교합니다.

## 학습 목표

1. A\* vs Hybrid A\* 상태 공간 차이
2. Bicycle Model 운동 방정식
3. (ix,iy,iyaw) 버킷으로 연속 상태 이산화
4. 격자 꺾임 경로 vs Hybrid 부드러운 경로 비교

In [ ]:
%matplotlib inline
import sys, os, math, heapq, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from utils.grid_map import make_simple_map
from utils.vehicle_model import plot_car, WB

ox, oy, sx, sy, gx, gy = make_simple_map()
print(f'환경 로드 완료 — 차량 축간거리 WB = {WB} m')
print(f'출발 ({sx},{sy})  ->  목표 ({gx},{gy})')


## 1. A\*와 Hybrid A\*의 차이

### 이론 배경

| 구분 | A\* | Hybrid A\* |
|------|------|----------|
| 상태 | 격자 (ix,iy) | 연속 (x,y,yaw) |
| 이동 | 8방향 격자 | Bicycle Model |
| 키 | (ix,iy) | (ix,iy,iyaw) |
| 경로 | 꺾임 | 부드러움(주행 가능) |

Heuristic은 동일(Euclidean). **이웃 확장 방식**만 다릅니다.

## 2. Bicycle Model 운동 방정식

### 이론 배경

$$x' = x + L\cos\theta,\quad y' = y + L\sin\theta,\quad \theta' = \theta + \frac{L\tan\delta}{WB}$$

$L$=이동거리, $\delta$=조향각, $WB$=축간거리. 차량의 최소 회전반경 제약을 반영합니다.

In [ ]:
def bicycle_step(x, y, yaw, steer, step_size=2.0, L=WB):
    # 전륜 조향 Bicycle Model 한 스텝
    x   += step_size * math.cos(yaw)
    y   += step_size * math.sin(yaw)
    yaw += step_size * math.tan(steer) / L
    yaw  = (yaw + math.pi) % (2*math.pi) - math.pi   # -pi~pi 유지
    return x, y, yaw

print('조향각별 1스텝 결과 (출발 원점/직진, step=2m)')
print(f'  {"steer":>7}  {"x":>6}  {"y":>6}  {"yaw_new":>9}')
for d in [-40, -20, 0, 20, 40]:
    nx, ny, nyaw = bicycle_step(0, 0, 0, math.radians(d))
    print(f'  {d:>6}deg  {nx:>6.2f}  {ny:>6.2f}  {math.degrees(nyaw):>7.1f}deg')


## 3. 조향각별 궤적

### 이론 배경

한 조향각을 계속 유지하면 곡선 궤적이 됩니다. 5가지 조향각을 한 스텝씩 그려 비교합니다.

In [ ]:
# 각 조향각을 계속 유지하면 곡선 궤적이 됨 (한 스텝씩 그려가며 확인)
STEERS = [-40, -25, 0, 25, 40]
COLORS = ['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd']
N = 15
trajs = []
for d in STEERS:
    x, y, yaw = 0.0, 0.0, 0.0; xs, ys = [0.0], [0.0]
    for _ in range(N):
        x, y, yaw = bicycle_step(x, y, yaw, math.radians(d)); xs.append(x); ys.append(y)
    trajs.append((xs, ys))

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(-2, 33); ax.set_ylim(-20, 20); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.plot(0, 0, 'og', ms=10); ax.set_title('Bicycle Model: constant-steering trajectories')
lines = [ax.plot([], [], '-', color=c, lw=2, label=f'{d}deg')[0] for c, d in zip(COLORS, STEERS)]
ax.legend(loc='upper left', fontsize=8)

def upd(f):
    for ln, (xs, ys) in zip(lines, trajs): ln.set_data(xs[:f+2], ys[:f+2])
    return lines
HTML(FuncAnimation(fig, upd, frames=N, interval=200, repeat=False, blit=False).to_jshtml())


## 4. HybridNode 클래스

### 이론 배경

연속 상태 (x,y,yaw)를 저장하되, 중복 탐색 방지를 위해 **이산 버킷 인덱스 (ix,iy,iyaw)** 도 함께 가집니다.  
같은 버킷이면 이미 방문한 상태로 취급합니다.

In [ ]:
RESOLUTION = 2.0
YAW_RES    = math.radians(15.0)   # yaw를 15도 단위로 이산화 (버킷 24개)

class HybridNode:
    # 연속 상태 (x,y,yaw) + 이산 버킷 인덱스 (ix,iy,iyaw)
    def __init__(self, x, y, yaw, cost, parent_index):
        self.x, self.y, self.yaw = x, y, yaw
        self.cost = cost
        self.parent_index = parent_index
        self.ix = round(x / RESOLUTION)
        self.iy = round(y / RESOLUTION)
        self.iyaw = round(yaw / YAW_RES)
    def index(self):
        return (self.ix, self.iy, self.iyaw)   # dict 키 (중복 제거용)
    def __lt__(self, other):
        return self.cost < other.cost

hn = HybridNode(5.0, 3.0, math.radians(30), 2.5, None)
print(f'예시: x={hn.x}, y={hn.y}, yaw=30deg -> 버킷 키 index()={hn.index()}')
print('A*는 키가 (ix,iy) 2D, Hybrid A*는 (ix,iy,iyaw) 3D (방향까지 구분)')


## 5. 상태 이산화 버킷

### 이론 배경

yaw는 15도 단위 24개 버킷으로, 위치는 격자 셀로 묶입니다.  
비슷한 상태를 같은 버킷으로 모아 무한 탐색을 방지합니다.

In [ ]:
# 연속 상태가 어떻게 이산 버킷으로 묶이는지 (왜 중복 탐색이 안 생기는지)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

ax = axes[0]   # yaw 이산화: 24개 방향 버킷
n_buckets = int(round(2*math.pi / YAW_RES))
for b in range(n_buckets):
    ang = b * YAW_RES
    ax.annotate('', xy=(math.cos(ang), math.sin(ang)), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=plt.cm.hsv(b/n_buckets), lw=1.5, alpha=0.8))
for deg, mk in [(30,'o'), (37,'s')]:
    yr = math.radians(deg); ax.plot(0.7*math.cos(yr), 0.7*math.sin(yr), mk, color='black', ms=10, zorder=5)
ax.set_xlim(-1.3,1.3); ax.set_ylim(-1.3,1.3); ax.set_aspect('equal')
ax.text(0, -1.2, '30deg & 37deg -> same bucket', ha='center', fontsize=8, color='red')
ax.set_title(f'Yaw -> {n_buckets} buckets (15deg each)'); ax.grid(True, alpha=0.3)

ax2 = axes[1]   # (ix,iy) 격자 버킷
for g in range(7): ax2.axvline(g*RESOLUTION, color='lightgray', lw=0.8); ax2.axhline(g*RESOLUTION, color='lightgray', lw=0.8)
for px, py in [(3.0,5.0), (3.8,5.5), (2.5,4.6)]:
    ax2.plot(px, py, 'o', color='navy', ms=9)
    ax2.plot(round(px/RESOLUTION)*RESOLUTION, round(py/RESOLUTION)*RESOLUTION, 's', color='red', ms=12, alpha=0.4)
ax2.set_aspect('equal'); ax2.set_xlim(0,12); ax2.set_ylim(0,12)
ax2.text(6, 0.5, 'nearby (x,y) -> same (ix,iy)', ha='center', fontsize=8, color='red')
ax2.set_title('Position -> grid cell (ix,iy)')
plt.suptitle('State Discretization: index() = (ix, iy, iyaw)', fontsize=12)
plt.tight_layout(); plt.show()
print('같은 버킷에 들어온 상태는 이미 방문한 것으로 보고 중복 탐색하지 않는다')


## 6. 이웃 확장 (조향 후보)

### 이론 배경

A\*의 8방향 대신, 조향각 -40~+40도(15도 간격)에 Bicycle Model을 적용해 이웃을 만듭니다.  
비용에 `|조향각|×0.1` 페널티를 줘 급조향을 억제합니다.

In [ ]:
STEER_RES = math.radians(15.0); MAX_STEER = math.radians(40.0); STEP_SIZE = 2.0

def get_neighbors(node):
    # 조향각 -40~+40도(15도 간격)로 다음 상태 생성. 비용 = 이동거리 + |조향|*0.1
    neighbors = []
    steer = -MAX_STEER
    while steer <= MAX_STEER + 1e-6:
        nx, ny, nyaw = bicycle_step(node.x, node.y, node.yaw, steer, STEP_SIZE)
        neighbors.append(HybridNode(nx, ny, nyaw, node.cost + STEP_SIZE + abs(steer)*0.1, node.index()))
        steer += STEER_RES
    return neighbors

nbs = get_neighbors(HybridNode(0, 0, 0, 0, None))
print(f'출발점에서 가능한 다음 상태: {len(nbs)}개 (조향각 후보 수)')


In [ ]:
# 조향 후보를 여러 단계 조합하면 도달 가능 영역이 부채꼴로 퍼진다 (A*의 8격자 이동과 대조)
steers = list(np.arange(-MAX_STEER, MAX_STEER+1e-6, STEER_RES))
fig, ax = plt.subplots(figsize=(8, 6))

def expand(state, depth, max_depth):
    x, y, yaw = state
    if depth >= max_depth: return
    for steer in steers:
        nx, ny, nyaw = bicycle_step(x, y, yaw, steer, STEP_SIZE)
        ax.plot([x, nx], [y, ny], '-', color='steelblue', lw=0.8, alpha=max(0.7-0.18*depth, 0.15))
        ax.plot(nx, ny, '.', color='navy', ms=3, alpha=0.5)
        expand((nx, ny, nyaw), depth+1, max_depth)

expand((0.0, 0.0, 0.0), 0, 4)
ax.plot(0, 0, 'og', ms=12, zorder=5, label='Start')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
ax.set_title('Reachable Tree (4 steps deep) — only car-feasible curves')
plt.tight_layout(); plt.show()
print('차량 기구학을 만족하는 곡선만 생성 -> 실제 주행 가능한 경로')


## 7. hybrid_astar 구현

### 이론 배경

A\* 골격 그대로, dict 키를 `node.index()`로, 이웃 확장을 `get_neighbors()`로 바꿉니다.  
정확한 목표 도달이 어려우므로 **목표 반경 내** 도달하면 종료합니다.

In [ ]:
def heuristic_h(node, gx, gy):
    return math.hypot(node.x - gx, node.y - gy)   # 목표까지 유클리드 거리

def is_valid_cont(x, y, ox, oy, robot_radius=1.5):
    # 연속 좌표가 모든 장애물에서 robot_radius 이상 떨어졌는지
    return all(math.hypot(iox-x, ioy-y) > robot_radius for iox, ioy in zip(ox, oy))

print('heuristic_h / is_valid_cont 정의 완료')


In [ ]:
def hybrid_astar(sx, sy, syaw, gx, gy, ox, oy, robot_radius=1.5):
    start = HybridNode(sx, sy, syaw, 0.0, None)
    open_set = {start.index(): start}; closed_set = {}
    pq = [(heuristic_h(start, gx, gy), start)]
    best, best_h = start, float('inf')

    for _ in range(20000):
        if not pq: break
        _, current = heapq.heappop(pq)
        if current.index() not in open_set: continue
        open_set.pop(current.index()); closed_set[current.index()] = current
        h = heuristic_h(current, gx, gy)
        if h < best_h: best_h, best = h, current
        if h < STEP_SIZE * 1.5: break                 # 목표 반경 내 도달
        for nb in get_neighbors(current):             # Bicycle Model 확장
            if not is_valid_cont(nb.x, nb.y, ox, oy, robot_radius): continue
            if nb.index() in closed_set: continue
            if nb.index() not in open_set or open_set[nb.index()].cost > nb.cost:
                open_set[nb.index()] = nb
                heapq.heappush(pq, (nb.cost + heuristic_h(nb, gx, gy), nb))   # f = g + h

    path = []; node = best                            # 역추적 (parent 객체 체인)
    while node is not None:
        path.append(node)
        node = closed_set.get(node.parent_index) if node.parent_index is not None else None
    path.reverse()
    return path, closed_set

print('hybrid_astar() 정의 완료 — A* 골격에 이웃 확장만 Bicycle Model로 교체')


## 8. 탐색 실행

### 이론 배경

출발 자세 45도로 탐색합니다. 탐색 노드 수가 (ix,iy,iyaw) 3D 버킷이라 A\*보다 많을 수 있습니다.

In [ ]:
print('Hybrid A* 탐색 중...')
t0 = time.time()
path, closed_set = hybrid_astar(sx, sy, math.radians(45), gx, gy, ox, oy)
dt = (time.time()-t0)*1000
path_x = [n.x for n in path]; path_y = [n.y for n in path]; path_yaw = [n.yaw for n in path]
print(f'탐색 노드 {len(closed_set):,}개,  경로 {len(path)}개 점,  {dt:.0f} ms')
print(f'목표 도달 거리 {math.hypot(path_x[-1]-gx, path_y[-1]-gy):.2f} m')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(ox, oy, '.k', ms=2, label='Obstacle')
ax.plot(sx, sy, 'og', ms=10, label='Start'); ax.plot(gx, gy, 'xb', ms=10, mew=2, label='Goal')
ax.plot([n.x for n in closed_set.values()], [n.y for n in closed_set.values()],
        '.', color='cyan', ms=2, alpha=0.4, label=f'Explored ({len(closed_set):,})')
ax.plot(path_x, path_y, '-r', lw=2, label='Path')
for i in range(0, len(path), max(1, len(path)//8)):
    plot_car(path_x[i], path_y[i], path_yaw[i], ax=ax, color='royalblue', alpha=0.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
ax.set_title('Hybrid A* — (x,y,yaw) Path with Vehicle Poses')
plt.tight_layout(); plt.show()


## 9. 격자 경로 vs Hybrid 경로 비교

### 이론 배경

같은 출발/목표에서 격자 8방향 경로(왼쪽)와 Hybrid 경로(오른쪽)를 비교합니다.  
격자 경로는 급격한 꺾임이 있어 차량이 그대로 못 따라가지만, Hybrid 경로는 최소 회전반경을 지키는 부드러운 곡선입니다.

In [ ]:
# A*(격자 8방향) 경로 vs Hybrid A*(연속) 경로 비교
from utils.grid_search import dijkstra_search
class GridNode:
    def __init__(s, x, y, c, p): s.x, s.y, s.cost, s.parent_index = x, y, c, p
    def __lt__(s, o): return s.cost < o.cost
gpx, gpy, *_ = dijkstra_search(sx, sy, gx, gy, ox, oy, 2.0, 1.0, GridNode)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax in axes:
    ax.plot(ox, oy, '.k', ms=2); ax.plot(sx, sy, 'og', ms=10); ax.plot(gx, gy, 'xb', ms=10, mew=2)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
axes[0].plot(gpx, gpy, '-o', color='purple', ms=3, lw=1.5, label='Grid path (8-dir)')
axes[0].set_title('A* / Dijkstra — jagged grid path'); axes[0].legend(fontsize=8)
axes[1].plot(path_x, path_y, '-r', lw=2, label='Hybrid path')
for i in range(0, len(path), max(1, len(path)//8)):
    plot_car(path_x[i], path_y[i], path_yaw[i], ax=axes[1], color='royalblue', alpha=0.5)
axes[1].set_title('Hybrid A* — smooth, car-feasible'); axes[1].legend(fontsize=8)
plt.suptitle('Grid path (corners) vs Hybrid A* (smooth, yaw-aware)', fontsize=13)
plt.tight_layout(); plt.show()
print('격자 경로: 급격한 꺾임 -> 차량이 그대로 못 따라감')
print('Hybrid 경로: 최소 회전반경을 지키는 부드러운 곡선 -> 주차/저속 정밀 주행에 적합')
print('Day 1 완료! 다음 Day 2 — Global Path를 주행 가능한 Trajectory로 변환')
